# Essential Exploratory Data Analysis (EDA) Pandas Methods

We have used many different Pandas attributes and methods over the last two weeks. Let's review the most essential ones that you will use when you BEGIN exploring data.

## Import Modules

In [ ]:
# Import essential libraries for data manipulation and math
import numpy as np
import pandas as pd

## Read data

Let's continue to work with the JOINED data set that we created previously.

In [ ]:
# Load the dataset into a Pandas DataFrame
df = pd.read_csv('data/joined_data.csv')

In [ ]:
# Preview the entire DataFrame (useful for small datasets)
df

But we CANNOT look at a dataset that has thousands to hundreds of thousands or even millions of rows!

We cannot look at a data set that has dozens to hundreds of columns!

What are the basic actions that we should perform for ANY data analysis task?

## Exploratory Data Analysis (EDA)

In [ ]:
# Get the dimensions of the DataFrame (rows, columns)
df.shape

In [ ]:
# List all the column headers present in the DataFrame
df.columns

In [ ]:
# Check the data type (dtype) of each column
df.dtypes

In [ ]:
# Print a concise summary of the DataFrame (index, non-null counts, dtypes, memory usage)
df.info()

In [ ]:
# Count how many columns belong to each data type
df.dtypes.value_counts()

In [ ]:
# Calculate the total number of missing (NaN) values per column
df.isna().sum()

In [ ]:
# Filter to see ONLY the columns that have ZERO missing values
df.isna().sum()[ df.isna().sum() < 1 ]

In [ ]:
# Filter to see ONLY the columns that DO have missing values (count > 0)
df.isna().sum()[ df.isna().sum() > 0 ]

In [ ]:
# Count the number of unique elements in each column
df.nunique()

`.nunique()` method does NOT treat MISSINGS as a VALUE. By default:

In [ ]:
# Explicitly drop NaNs from unique count (this is the default behavior)
df.nunique(dropna=True)

If you switch `dropna=False` then the MISSING is counted as a VALUE.

In [ ]:
# Include NaNs as a distinct unique value category
df.nunique(dropna=False)

I think it is useful to examine for columns with 1 or 2 unique values (potential booleans or constants).

Lastly, it is always important to begin summarizing the columns. 

**Watch out:** If we try to calculate the mean of the entire DataFrame, Pandas might throw a `TypeError` if it tries to mathematically average string/object columns!

In [ ]:
# This will intentionally raise a TypeError in newer Pandas versions because column 'A' contains strings!
df.mean()

To fix this, we must tell Pandas to only calculate the mean for numerical columns using `numeric_only=True`.

In [ ]:
# Calculate column means exclusively for int/float columns
df.mean(numeric_only=True)

In [ ]:
# Generate descriptive statistics for numerical columns (count, mean, std, min, max, quartiles)
df.describe()

In [ ]:
# Generate descriptive statistics strictly for string/categorical columns (count, unique, top, freq)
df.describe(include='object')

In [ ]:
# Generate descriptive statistics for ALL columns mixed together (results in some NaNs where stats don't apply)
df.describe(include='all')

The next step is to begin counting the categorical/string columns!

In [ ]:
# Get a frequency count of each unique value in column A
df.A.value_counts()

In [ ]:
# Get frequency counts, ensuring missing values (NaN) are included in the breakdown
df.A.value_counts(dropna=False)

In [ ]:
# Same operation on column D
df.D.value_counts(dropna=False)

In [ ]:
# Same operation on column E
df.E.value_counts(dropna=False)

In [ ]:
# Use normalize=True to return relative frequencies (percentages) instead of raw counts
df.E.value_counts(dropna=False, normalize=True)

In [ ]:
df.H.value_counts(dropna=False)

In [ ]:
df.H.value_counts(dropna=False, normalize=True)

In [ ]:
df.F.value_counts()

In [ ]:
df.F.value_counts(dropna=False, normalize=True)

## Realistic example

Let's follow the same steps to get the same type of BASIC information on a real data set!

In [ ]:
!pip install seaborn --break-system-packages

In [ ]:
# Seaborn has several built-in datasets perfect for practice
import seaborn as sns

In [ ]:
# Load the classic Titanic dataset
titanic = sns.load_dataset('titanic')

In [ ]:
titanic.shape

In [ ]:
titanic.columns

In [ ]:
titanic

In [ ]:
# Preview the first 5 rows
titanic.head(10)

In [ ]:
# Preview the last 5 rows
titanic.tail()

In [ ]:
titanic.dtypes

In [ ]:
titanic.dtypes.value_counts()

In [ ]:
titanic.info()

In [ ]:
titanic.isna().sum()

In [ ]:
# Find columns that contain at least one missing value
titanic.isna().sum()[ titanic.isna().sum() > 0 ]

In [ ]:
titanic.nunique()

In [ ]:
titanic.nunique(dropna=False)

In [ ]:
titanic.describe()

In [ ]:
# How many survived? (0 = No, 1 = Yes)
titanic.survived.value_counts()

In [ ]:
# What percentage survived?
titanic.survived.value_counts(dropna=False, normalize=True)

In [ ]:
titanic.describe(include='object')

In [ ]:
titanic.alive.value_counts()

In [ ]:
titanic.describe(include='category')

In [ ]:
titanic.describe(include='boolean')

In [ ]:
titanic.pclass.value_counts()

In [ ]:
titanic.pclass.value_counts(dropna=False, normalize=True)

In [ ]:
# 'class' is a reserved keyword in Python, so we must use bracket notation instead of dot notation here!
titanic['class'].value_counts()

In [ ]:
titanic['class'].value_counts(dropna=False, normalize=True)

In [ ]:
titanic.deck.value_counts()

In [ ]:
titanic.deck.value_counts(dropna=False)

In [ ]:
titanic.deck.value_counts(dropna=False, normalize=True)

# SPLIT-APPLY-COMBINE or GROUPBY and AGGREGATE (summarize)

We have learned how to summarize data in Pandas. But we have summarized INDIVIDUAL columns ignoring all other columns!

Let's now explore how summary stats of one column CHANGE or VARY across the categories of another column!

Or...is the average different for a different group?

## Read data

Continue working with our JOINED data set.

In [ ]:
df = pd.read_csv('data/joined_data.csv')
df.info()

## Review

We know how to summarize individual columns!

In [ ]:
df.nunique()

In [ ]:
df.E.value_counts(dropna=False)

In [ ]:
# Global mean of B
df.B.mean()

In [ ]:
# Global mean of C
df.C.mean()

But what is the AVERAGE of `B` for each unique category of `E`?

We cannot answer this question by simply applying the same summary methods globally!

We need to do something else to support our exploration!

## Split-Apply-Combine (Manual Method)

**Split** means we DIVIDE or BREAK the data into distinct and separate groups!

We must partition the data set into the unique categories of a **GROUPING VARIABLE**.

We need to know the unique values of the grouping variable.

In [ ]:
# Find all unique categories we need to split by
df.E.unique()

We must split `df` into smaller data sets. Each data set will have 1 and only 1 value of `E`.

In [ ]:
# SPLIT: Create a sub-dataframe strictly for category 'aa'
df_E_aa = df.loc[ df.E == 'aa', : ].copy()

In [ ]:
df_E_aa

We can apply all Pandas methods to our smaller partitioned DataFrame!

In [ ]:
# APPLY: Calculate mean of B just for 'aa' group
df_E_aa.B.mean()

We need to repeat the SPLITTING process for each unique value of `E`.

In [ ]:
# Split for 'bb', 'cc', 'dd'
df_E_bb = df.loc[ df.E == 'bb', : ].copy()
df_E_cc = df.loc[ df.E == 'cc', : ].copy()
df_E_dd = df.loc[ df.E == 'dd', : ].copy()

WE also need to SPLIT on the MISSING values of `E`.

In [ ]:
# Using .isna() to isolate rows where E is missing
df_E_na = df.loc[ df.E.isna(), :].copy()

We need to APPLY the summary method to `B` within each SPLIT smaller data set!

In [ ]:
df_E_bb.B.mean()

In [ ]:
df_E_cc.B.mean()

In [ ]:
df_E_dd.B.mean()

In [ ]:
df_E_na.B.mean()

But, we do NOT want to just let these values stay as stray numbers in our notebook.

We want to **COLLECT** or **COMBINE** the summary statistics PER GROUP into a new DataFrame!!!

SPLIT-APPLY-COMBINE breaks a dataset based on categories, applies methods to summarize variables within each smaller dataset, and then combines the summary statistics per group into a new easy-to-use dataframe!

In [ ]:
# COMBINE: Manually stitch the results back into a DataFrame
df_E_summary = pd.DataFrame({'E': df.E.unique(),
                             'B_avg': [df_E_aa.B.mean(), df_E_bb.B.mean(), df_E_cc.B.mean(), df_E_dd.B.mean(), df_E_na.B.mean()]})

In [ ]:
df_E_summary

## Enter `.groupby()`
Pandas has a native method to manage SPLIT-APPLY-COMBINE for us instantly!!!!

The `.groupby()` method will automatically divide the dataset into smaller groups based on the categories of a grouping variable!!!

In [ ]:
# Does exactly what we just did manually, in one line of code!
df.groupby('E').B.mean()

We can force `.groupby()` to INCLUDE the MISSING (which is dropped by default)!

In [ ]:
df.groupby('E', dropna=False).B.mean()

You can also use bracket notation instead of dot notation, which is helpful if column names have spaces or if you're using string variables to define your columns dynamically.

In [ ]:
var_to_group = 'E'
var_to_summarize = 'B'
df.groupby(var_to_group, dropna=False)[var_to_summarize].mean()

We can apply different standard operations over the groups:

In [ ]:
# Standard Deviation
df.groupby('E', dropna=False).B.std()

In [ ]:
# Standard Error of the Mean
df.groupby('E', dropna=False).B.sem()

If we want MULTIPLE summary stats returned, we can apply the `.describe()` method directly on the groupby object!

In [ ]:
df.groupby('E', dropna=False).B.describe()

We do not need to necessarily identify a single column...we can group the whole dataframe! (Remember to pass `numeric_only=True` to avoid string aggregation errors).

In [ ]:
df.groupby('E', dropna=False).mean(numeric_only=True)

In [ ]:
df.groupby('E', dropna=False).describe()

## Named Aggregations `.aggregate()`
Pandas provides multiple different ways to APPLY summary methods to GROUPED dataframes.

I personally like the `.groupby().aggregate()` approach. I feel this is the most flexible yet straightforward way to apply summary methods to DIFFERENT COLUMNS simultaneously!!!

The syntax looks like this: `New_Column_Name = ('Original_Column', 'statistic_to_calculate')`

In [ ]:
df_E_summary_info = df.groupby('E', dropna=False).\
aggregate(B_avg = ('B', 'mean'))
df_E_summary_info

In [ ]:
df_E_summary_info = df.groupby('E', dropna=False).\
aggregate(B_avg = ('B', 'mean'),
          B_std = ('B', 'std'))
df_E_summary_info

In [ ]:
# The backslash '\' allows us to break a long line of code across multiple lines for readability
df_E_summary_info = df.groupby('E', dropna=False).\
aggregate(B_avg = ('B', 'mean'),
          B_std = ('B', 'std'),
          B_sem = ('B', 'sem'),
          C_avg = ('C', 'mean'),
          C_sem = ('C', 'sem'),
          B_numrows = ('B', 'size'),
          B_nonmissing = ('B', 'count'),
          F_nunique = ('F', 'nunique'),
          G_nunique = ('G', 'nunique'),
          A_nunique = ('A', 'nunique')).\
reset_index() # Pulls 'E' out of the index and back into a standard column!

In [ ]:
df_E_summary_info

We can also group by MULTIPLE COLUMNS if we supply multiple column names within a list!

In [ ]:
# Grouping by E AND H
df.groupby(['E', 'H'], dropna=False).\
aggregate(B_avg = ('B', 'mean'),
          B_numrows = ('B', 'size'),
          B_nonmissing = ('B', 'count')).\
reset_index()

The multi-index (when you group by multiple items without `reset_index()`) is annoying and causes a lot of filtering issues. So again, I highly recommend and encourage resetting the index!!

## Realistic Groupby Example

In [ ]:
# Re-load Titanic dataset to practice aggregations
titanic = sns.load_dataset('titanic')

In [ ]:
# Global survival rate
titanic.survived.mean()

But does the survival rate depend on the passenger class?

Let's GROUP BY `pclass` and APPLY methods to SUMMARIZE the `survived` column.
This way we can explore if the survival rate changes across the `pclass` categories!

In [ ]:
titanic.groupby('pclass', dropna=False).\
aggregate(num_rows = ('survived', 'size'),
          num_nonmissing = ('survived', 'count'),
          num_survive = ('survived', 'sum'),
          prop_survive = ('survived', 'mean'),
          survive_sem = ('survived', 'sem')).\
reset_index()

Let's group by 2 variables to drill down even further. Let's group by `pclass` and `class` (which are essentially capturing the same hierarchy here, but useful to see cross-tabulations).

In [ ]:
titanic.groupby(['pclass', 'class'], dropna=False).\
aggregate(num_rows = ('survived', 'size'),
          num_nonmissing = ('survived', 'count'),
          num_survive = ('survived', 'sum'),
          prop_survive = ('survived', 'mean'),
          survive_sem = ('survived', 'sem')).\
reset_index()

It can be useful to simplify the grouped and summarized result by only focusing on the OBSERVED combinations!
If you pass `observed=True`, Pandas will drop unobserved empty group combinations (like pclass 1 intersecting with Third class).

In [ ]:
titanic.groupby(['pclass', 'class'], dropna=False, observed=True).\
aggregate(num_rows = ('survived', 'size'),
          num_nonmissing = ('survived', 'count'),
          num_survive = ('survived', 'sum'),
          prop_survive = ('survived', 'mean'),
          survive_sem = ('survived', 'sem')).\
reset_index()

In [ ]:
titanic.groupby(['pclass', 'class'], dropna=False, observed=False).\
aggregate(num_rows = ('survived', 'size'),
          num_nonmissing = ('survived', 'count'),
          num_survive = ('survived', 'sum'),
          prop_survive = ('survived', 'mean'),
          survive_sem = ('survived', 'sem')).\
reset_index()